# 06. Actor-Critic Methods (A2C/A3C)

**Nivel:** 🔴 Avanzado  
**Tiempo estimado:** 90 minutos  
**Prerequisitos:** [05. Policy Gradients](05-policy-gradients.ipynb)

## 🎯 Objetivos de Aprendizaje
Al finalizar este notebook, podrás:
- Comprender cómo Actor-Critic combina value-based y policy-based methods
- Implementar el algoritmo Advantage Actor-Critic (A2C)
- Entender la función de ventaja y su rol en reducir varianza
- Aplicar actor-critic a problemas complejos
- Comprender A3C (Asynchronous Advantage Actor-Critic) y paralelización

## 📚 Motivación

### El Mejor de Dos Mundos

Hemos visto dos familias de algoritmos:

**Value-Based (DQN)**:
- ✅ Bajo varianza
- ✅ Eficiente con muestras
- ❌ Solo acciones discretas
- ❌ Puede divergir

**Policy-Based (REINFORCE)**:
- ✅ Acciones continuas
- ✅ Convergencia garantizada
- ❌ Alta varianza
- ❌ Ineficiente con muestras

### La Solución: Actor-Critic

**Idea**: Usar DOS redes neuronales:
1. **Actor**: Política π(a|s; θ) que selecciona acciones
2. **Critic**: Función de valor V(s; w) que evalúa estados

El critic reduce la varianza del actor proporcionando un baseline aprendido.

### Ventajas de Actor-Critic

- Menor varianza que REINFORCE (usa critic como baseline)
- Más eficiente con muestras (usa TD learning)
- Funciona con acciones continuas
- Base de algoritmos state-of-the-art (PPO, SAC, TD3)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import gymnasium as gym
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.distributions import Categorical
import pandas as pd

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Librerías importadas | Dispositivo: {device}")

## 📐 Fundamentos Matemáticos

### Función de Ventaja

La **ventaja** mide qué tan buena es una acción comparada con el promedio:

$$A(s, a) = Q(s, a) - V(s)$$

- Si $A > 0$: La acción es mejor que el promedio
- Si $A < 0$: La acción es peor que el promedio

### Estimación de la Ventaja

Usando TD(0):
$$A(s_t, a_t) \approx r_t + \gamma V(s_{t+1}) - V(s_t) = \delta_t$$

donde $\delta_t$ es el error TD.

### Algoritmo Actor-Critic

**Actor Update** (Policy Gradient):
$$\theta \leftarrow \theta + \alpha_\theta \nabla_\theta \log \pi_\theta(a_t|s_t) A(s_t, a_t)$$

**Critic Update** (TD Learning):
$$w \leftarrow w + \alpha_w \delta_t \nabla_w V(s_t; w)$$

### Pseudocódigo A2C

```
Inicializar actor π(a|s; θ) y critic V(s; w)
Para cada episodio:
    Inicializar s
    Para cada paso:
        Seleccionar a ~ π(·|s; θ)
        Ejecutar a, observar r, s'
        
        # Calcular ventaja
        δ = r + γ V(s'; w) - V(s; w)
        
        # Actualizar critic
        w ← w + αw δ ∇w V(s; w)
        
        # Actualizar actor
        θ ← θ + αθ δ ∇θ log π(a|s; θ)
        
        s ← s'
```

## 💻 Implementación

In [ ]:
class ActorCriticNetwork(nn.Module):
    """Red compartida para actor y critic."""
    
    def __init__(self, state_dim, action_dim, hidden_dim=128):
        super().__init__()
        # Capas compartidas
        self.shared = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU()
        )
        
        # Actor head (política)
        self.actor = nn.Linear(hidden_dim, action_dim)
        
        # Critic head (función de valor)
        self.critic = nn.Linear(hidden_dim, 1)
    
    def forward(self, state):
        features = self.shared(state)
        policy_logits = self.actor(features)
        value = self.critic(features)
        return F.softmax(policy_logits, dim=-1), value

class A2CAgent:
    """Advantage Actor-Critic Agent."""
    
    def __init__(self, state_dim, action_dim, lr=1e-3, gamma=0.99,
                 value_coef=0.5, entropy_coef=0.01):
        self.gamma = gamma
        self.value_coef = value_coef
        self.entropy_coef = entropy_coef
        
        self.network = ActorCriticNetwork(state_dim, action_dim).to(device)
        self.optimizer = optim.Adam(self.network.parameters(), lr=lr)
        
        self.history = {'episode': [], 'reward': []}
    
    def select_action(self, state):
        """Selecciona acción y retorna log_prob, value."""
        state = torch.FloatTensor(state).unsqueeze(0).to(device)
        probs, value = self.network(state)
        m = Categorical(probs)
        action = m.sample()
        return action.item(), m.log_prob(action), m.entropy(), value
    
    def update(self, log_prob, value, reward, next_value, done):
        """Actualización online (cada paso)."""
        # Calcular ventaja usando TD
        if done:
            advantage = reward - value
        else:
            advantage = reward + self.gamma * next_value - value
        
        # Loss del actor (policy gradient)
        actor_loss = -(log_prob * advantage.detach())
        
        # Loss del critic (MSE del error TD)
        critic_loss = advantage.pow(2)
        
        # Loss total
        total_loss = actor_loss + self.value_coef * critic_loss
        
        # Backpropagation
        self.optimizer.zero_grad()
        total_loss.backward()
        self.optimizer.step()
        
        return total_loss.item()
    
    def train(self, env, n_episodes=500):
        """Entrena el agente."""
        for ep in range(n_episodes):
            state, _ = env.reset()
            total_reward = 0
            
            for t in range(500):
                # Seleccionar acción
                action, log_prob, entropy, value = self.select_action(state)
                
                # Ejecutar acción
                next_state, reward, terminated, truncated, _ = env.step(action)
                done = terminated or truncated
                total_reward += reward
                
                # Obtener valor del siguiente estado
                if done:
                    next_value = torch.tensor([[0.0]]).to(device)
                else:
                    with torch.no_grad():
                        next_state_tensor = torch.FloatTensor(next_state).unsqueeze(0).to(device)
                        _, next_value = self.network(next_state_tensor)
                
                # Actualizar redes
                self.update(log_prob, value, reward, next_value, done)
                
                if done:
                    break
                
                state = next_state
            
            self.history['episode'].append(ep)
            self.history['reward'].append(total_reward)
            
            if (ep + 1) % 50 == 0:
                avg = np.mean(self.history['reward'][-50:])
                print(f"Episodio {ep+1} | Reward promedio: {avg:.2f}")

print("✅ A2C implementado")

### Entrenamiento en CartPole

In [ ]:
env = gym.make('CartPole-v1')
agent = A2CAgent(env.observation_space.shape[0], env.action_space.n)

print("🎯 Entrenando A2C en CartPole\n")
agent.train(env, n_episodes=500)

# Visualizar
rewards_smooth = pd.Series(agent.history['reward']).rolling(20, min_periods=1).mean()
fig = go.Figure()
fig.add_trace(go.Scatter(x=agent.history['episode'], y=agent.history['reward'], 
                         mode='lines', name='Reward', opacity=0.3))
fig.add_trace(go.Scatter(x=agent.history['episode'], y=rewards_smooth,
                         mode='lines', name='Media móvil (20)'))
fig.update_layout(title='A2C Training on CartPole', 
                  xaxis_title='Episodio', yaxis_title='Reward',
                  template='plotly_white')
fig.show()

print(f"\n📊 Reward promedio final: {np.mean(agent.history['reward'][-50:]):.2f}")
env.close()

## 🔧 Versión con Framework

In [ ]:
from stable_baselines3 import A2C
from stable_baselines3.common.evaluation import evaluate_policy

env_sb3 = gym.make('CartPole-v1')

print("🎯 A2C con Stable-Baselines3\n")

model = A2C('MlpPolicy', env_sb3, verbose=1, learning_rate=7e-4)
model.learn(total_timesteps=50000)

mean_reward, std_reward = evaluate_policy(model, env_sb3, n_eval_episodes=100)
print(f"\n✅ Reward promedio: {mean_reward:.2f} ± {std_reward:.2f}")

env_sb3.close()

## 🎨 A3C: Asynchronous Advantage Actor-Critic

**A3C** extiende A2C con paralelización:

- Múltiples workers entrenan en paralelo
- Cada worker tiene su propio ambiente
- Comparten parámetros globales
- Acelera entrenamiento y estabiliza aprendizaje

### Arquitectura A3C

```
        Global Network (θ, w)
              ↓    ↓    ↓
         Worker1 Worker2 Worker3 ...
            Env1    Env2    Env3
              ↓      ↓      ↓
          Collect   Collect  Collect
         Experience Experience Experience
              ↓      ↓      ↓
           Update  Update  Update
            Global  Global  Global
```

**Nota**: A3C fue revolucionario en 2016, pero ahora A2C (síncrono) es más popular por ser más simple y similar rendimiento.

## 🎯 Ejercicios

### 🟢 Ejercicio 1: Tune Hiperparámetros
Experimenta con value_coef y entropy_coef.

In [ ]:
# TODO: Entrenar con diferentes coeficientes y comparar
pass

### 🟡 Ejercicio 2: GAE (Generalized Advantage Estimation)
Implementa GAE para mejor estimación de ventaja.

In [ ]:
# TODO: Implementar GAE
# Hint: GAE(λ) = Σ (γλ)^t δt
pass

### 🔴 Ejercicio 3: Comparación Completa
Compara DQN, REINFORCE, y A2C en el mismo ambiente.

In [ ]:
# TODO: Benchmark de los 3 algoritmos
pass

## 📚 Resumen

### Conceptos Clave

- **Actor-Critic**: Combina value-based y policy-based methods
- **Actor**: Aprende la política π(a|s; θ)
- **Critic**: Aprende la función de valor V(s; w)
- **Ventaja**: A(s,a) = Q(s,a) - V(s) reduce varianza
- **A2C**: Versión síncrona con múltiples workers
- **A3C**: Versión asíncrona paralela

### Evolución de Algoritmos RL

| Algoritmo | Tipo | Año | Característica Principal |
|-----------|------|-----|-------------------------|
| **Q-Learning** | Value-based | 1989 | Base tabular |
| **DQN** | Value-based | 2013 | Deep Q-learning |
| **REINFORCE** | Policy-based | 1992 | Policy gradient básico |
| **A3C** | Actor-Critic | 2016 | Paralelización |
| **PPO** | Actor-Critic | 2017 | Clipping para estabilidad |
| **SAC** | Actor-Critic | 2018 | Maximum entropy RL |

### Comparación Final

| Aspecto | Value-Based | Policy-Based | Actor-Critic |
|---------|-------------|--------------|---------------|
| **Varianza** | Baja | Alta | Media |
| **Eficiencia** | Alta | Baja | Media-Alta |
| **Acciones continuas** | ❌ | ✅ | ✅ |
| **Convergencia** | No garantizada | Garantizada | Generalmente estable |
| **Complejidad** | Media | Baja | Alta |

### ¿Qué Sigue?

Has completado la ruta de RL Clásico. Ahora puedes:

1. **Explorar algoritmos avanzados**:
   - PPO (Proximal Policy Optimization)
   - SAC (Soft Actor-Critic)
   - TD3 (Twin Delayed DDPG)

2. **Aplicar a problemas reales**:
   - Robótica (MuJoCo, PyBullet)
   - Juegos complejos (Atari, Unity ML-Agents)
   - Trading, optimización, control

3. **Continuar con LLM Agents**:
   - [Ruta 3: LLM Agents](../../03-llm-agents/)
   - Aplicar RL a optimización de prompts
   - Agentes autónomos con LLMs

## 🔗 Recursos Adicionales

### 📄 Papers Fundamentales

- **"Asynchronous Methods for Deep RL"** - Mnih et al. (2016)
  - Paper original de A3C
  - Demuestra poder de paralelización

- **"High-Dimensional Continuous Control Using GAE"** - Schulman et al. (2015)
  - Introduce Generalized Advantage Estimation
  - Mejora estimación de ventaja

- **"Proximal Policy Optimization"** - Schulman et al. (2017)
  - PPO, estado del arte actual
  - Más simple y robusto que A3C

### 📖 Recursos Adicionales

- Sutton & Barto - Capítulos 13-15
- Spinning Up in Deep RL - Actor-Critic section
- David Silver Lecture 7: Policy Gradient Methods

### 💻 Implementaciones

- Stable-Baselines3: https://stable-baselines3.readthedocs.io/
- RLlib (Ray): https://docs.ray.io/en/latest/rllib/
- CleanRL: https://github.com/vwxyzjn/cleanrl

---

<div align="center">

**🎉 ¡Felicitaciones! Has completado la Ruta de Reinforcement Learning Clásico 🎉**

Has dominado desde los fundamentos teóricos hasta algoritmos state-of-the-art.

**[← Volver al índice de la ruta](README.md) | [Ir a LLM Agents →](../../03-llm-agents/)**

</div>